# Scrapy & Anti-Bot — Lab Keamanan (Red Team vs Blue Team)

Notebook ini **mendidik keamanan**: kita bedah *bagaimana scraper sungguhan menembus proteksi
anti-bot* sebuah e-commerce nyata, **lalu** memetakan cara mendeteksi & mencegahnya. Tujuannya
bukan "biar bisa nyuri data", tapi supaya **engineer/blue-team paham serangannya** sehingga bisa
membangun pertahanan yang tepat.

Studi kasus: `https://www.blibli.com/c3/olahraga-sepeda/AK-1000051`

> Setiap teknik "merah" (offensive) di bawah selalu dipasangkan dengan bagian
> **"Pencegahan (Blue Team)"**. Itu inti pelajarannya.

## 0. Aturan Lab & Etika (wajib dibaca)

- **Hanya untuk edukasi & pertahanan.** Jangan dipakai untuk merugikan pihak lain.
- Kita hanya menyentuh **data yang publik** (tanpa login) dan **seminimal mungkin** request
  (beberapa saja), dengan jeda — **bukan** mass-harvest.
- Menembus proteksi anti-bot bisa **melanggar Terms of Service** dan, pada skala/akibat
  tertentu, **melanggar hukum** (mis. UU ITE, CFAA). Di dunia nyata: minta izin / pakai API
  resmi / data berlisensi.
- Notebook ini sengaja "berisik" menjelaskan pertahanan supaya kamu bisa **menutup celah** ini
  di sistemmu sendiri.

In [1]:
# --- Setup bersama untuk seluruh notebook ---
BROWSER_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
              "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
PAGE = "https://www.blibli.com/c3/olahraga-sepeda/AK-1000051"
CAT  = "AK-1000051"

def api_url(page=1, n=8):
    return ("https://www.blibli.com/backend/search/products"
            f"?categoryId={CAT}&page={page}&itemPerPage={n}&channelId=web")

# Header yang meniru Chrome (dipakai di beberapa level)
BROWSER_HEADERS = {
    "User-Agent": BROWSER_UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
    "sec-ch-ua": '"Chromium";v="124", "Google Chrome";v="124", "Not-A.Brand";v="99"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "Upgrade-Insecure-Requests": "1",
}
print("Setup siap. Target:", PAGE)

Setup siap. Target: https://www.blibli.com/c3/olahraga-sepeda/AK-1000051


## 1. Recon — "Boleh?" dan "Bisa?"

Sebelum menyerang/menscrape apa pun, profesional **selalu recon dulu**: cek `robots.txt`,
jenis halaman (statis vs SPA), dan jenis proteksi.

In [2]:
import requests
import urllib.robotparser as urobot
from protego import Protego  # parser robots.txt yang dipakai Scrapy

# Ambil robots.txt dengan UA browser (UA default Python sering kena 403 -> parser salah baca)
robots = requests.get("https://www.blibli.com/robots.txt",
                      headers={"User-Agent": BROWSER_UA}, timeout=20).text
rp = urobot.RobotFileParser(); rp.parse(robots.splitlines())
pr = Protego.parse(robots)

paths = {
    "Halaman kategori": "/c3/olahraga-sepeda/AK-1000051",
    "API produk (sumber data)": "/backend/search/products",
    "Halaman detail": "/p/nama-produk/ps--ABC-12345",
}
print(f"{'PATH':28} | urllib | Protego(Scrapy)")
print("-" * 60)
for label, path in paths.items():
    a = "BOLEH" if rp.can_fetch("*", path) else "DILARANG"
    b = "BOLEH" if pr.can_fetch(path, "*") else "DILARANG"
    print(f"{label:28} | {a:6} | {b}")

PATH                         | urllib | Protego(Scrapy)
------------------------------------------------------------
Halaman kategori             | BOLEH  | BOLEH
API produk (sumber data)     | BOLEH  | DILARANG
Halaman detail               | BOLEH  | BOLEH


Temuan: **API `/backend/search/*` `DILARANG`** (Protego/Scrapy benar; `urllib` keliru karena
lemah soal wildcard `*`). Padahal endpoint itulah sumber daftar produk. Artinya: secara
`robots.txt`, jalur datanya **tidak diizinkan**. (Catat ini untuk diskusi etika nanti.)

In [3]:
# Jenis halaman & jenis proteksi: lihat header & cookie respons
r = requests.get(PAGE, headers=BROWSER_HEADERS, timeout=20)
print("Halaman kategori  : HTTP", r.status_code, "|", len(r.text), "byte")
print("Server header     :", r.headers.get("Server"))
print("cf-ray (Cloudflare):", r.headers.get("cf-ray"))
print("Set-Cookie penting :", [c for c in r.cookies.keys()])
print("Produk di HTML?    : 'olahraga-sepeda' muncul",
      r.text.count("olahraga-sepeda"), "x (data diisi via API, bukan SSR)")

# Coba API langsung tanpa trik apa pun
a = requests.get(api_url(), headers={"User-Agent": BROWSER_UA, "Accept": "application/json"}, timeout=20)
print("\nAPI tanpa trik    : HTTP", a.status_code,
      "->", "DIBLOKIR" if a.status_code != 200 else "lolos")

Halaman kategori  : HTTP 200 | 186207 byte
Server header     : cloudflare
cf-ray (Cloudflare): a05a7f600cc7e791-CGK
Set-Cookie penting : ['__cf_bm', '_cfuvid']
Produk di HTML?    : 'olahraga-sepeda' muncul 0 x (data diisi via API, bukan SSR)



API tanpa trik    : HTTP 403 -> DIBLOKIR


**Profil target:**

- **SPA** (React) — daftar produk **tidak** ada di HTML, diisi via API setelah halaman dimuat.
- Di belakang **Cloudflare Bot Management** (lihat `cf-ray` + cookie `__cf_bm`).
- API data **403** untuk klien biasa, dan **`DILARANG` `robots.txt`**.

Inilah yang membuat orang sering bilang "Blibli nggak bisa di-scrape". Sebenarnya bisa —
dengan menumpuk teknik bypass. Mari bedah satu per satu (merah), lalu cara menutupnya (biru).

## 2. Tangga Eskalasi — Merah vs Biru

Tiap level menambah satu trik. Kita lihat **kapan API mulai tembus**, dan **apa pertahanannya**.

> ⚠️ **WAF itu *stateful*.** Cloudflare menilai berdasarkan IP + sesi + waktu. Karena sel recon
> tadi sudah "menghangatkan" IP kita, level menengah (L1/L2) **kadang sudah lolos** padahal dari
> IP "dingin" biasanya 403. Jadi **baca status aktual yang dicetak**, bukan hafalan — itu justru
> pelajaran pentingnya. Yang **selalu** konsisten: L0 (gagal) dan L3 (resep lengkap, berhasil).

### Level 0 — `requests` polos (UA Python)

Cara paling naif: langsung GET API. Inilah yang dilakukan bot pemula.

In [4]:
import requests
try:
    r = requests.get(api_url(), timeout=20)  # UA default: "python-requests/x"
    print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
except Exception as e:
    print("ERR", e)
print("=> Diblokir. UA 'python-requests' langsung dikenali.")

HTTP 403 | content-type: text/html; charset=UTF-8
=> Diblokir. UA 'python-requests' langsung dikenali.


**Pencegahan (Blue Team):** blokir/percepat-curigai **User-Agent non-browser** dan UA kosong;
ini deteksi paling murah. Jangan andalkan ini saja (gampang dipalsukan).

### Level 1 — Header browser lengkap

Scraper menambah `User-Agent` Chrome + `Accept`, `Accept-Language`, `sec-ch-ua`, dll.

In [5]:
r = requests.get(api_url(), headers={**BROWSER_HEADERS, "Accept": "application/json",
                                     "Referer": PAGE}, timeout=20)
ok = r.status_code == 200 and "json" in r.headers.get("content-type", "")
print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
print("=> Lolos: header browser membuat request tampak lebih sah." if ok
      else "=> Masih diblokir: header saja belum cukup, WAF cek lebih dari header.")
print("   (Ingat: WAF stateful — hasil bergantung 'kehangatan' IP/sesi saat ini.)")

HTTP 200 | content-type: application/json
=> Lolos: header browser membuat request tampak lebih sah.
   (Ingat: WAF stateful — hasil bergantung 'kehangatan' IP/sesi saat ini.)


**Pencegahan (Blue Team):** *header anomaly detection* — cek konsistensi (mis. `sec-ch-ua`
harus cocok dengan UA; urutan header browser punya pola khas). Tetap mudah dipalsukan, jadi
butuh lapisan berikutnya.

### Level 2 — Sidik jari TLS (JA3/JA4)

Rahasia utamanya: **`requests`/Scrapy punya "sidik jari" TLS khas Python** (urutan cipher,
ekstensi) yang **berbeda dari Chrome**. WAF seperti Cloudflare/Akamai mencocokkan JA3/JA4 ini —
sehingga walau UA-nya "Chrome", TLS-nya "Python" → **ketahuan bohong**.

Senjata merah: **`curl_cffi`** yang bisa **meniru TLS Chrome** (`impersonate="chrome"`).

In [6]:
from curl_cffi import requests as creq
# Impersonate TLS Chrome, tapi TANPA cookie warm-up dulu
r = creq.get(api_url(), impersonate="chrome",
             headers={"Referer": PAGE, "Accept": "application/json"}, timeout=20)
ok = r.status_code == 200 and "json" in r.headers.get("content-type", "")
print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
print("=> TLS Chrome menutup celah JA3 — makin sulit dibedakan dari browser asli." if ok
      else "=> TLS Chrome saja belum cukup di endpoint ini: butuh cookie warm-up (Level 3).")

HTTP 200 | content-type: application/json
=> TLS Chrome menutup celah JA3 — makin sulit dibedakan dari browser asli.


**Pencegahan (Blue Team):** **fingerprint TLS JA3/JA4** dan tolak/▲-tantang bila mismatch
(UA="Chrome" tapi TLS≠Chrome). Inilah yang membuat banyak scraper gagal — dan kenapa penyerang
beralih ke `curl_cffi`/`utls`/browser asli.

### Level 3 — Resep lengkap: warm-up cookie + TLS impersonate (andal TEMBUS)

Resep paling andal: **kunjungi halaman dulu** seperti browser sungguhan untuk **mendapat cookie
bot-management Cloudflare** (`__cf_bm`), lalu pakai cookie itu (dalam *session* yang sama,
dengan TLS Chrome) untuk memanggil API.

In [7]:
from curl_cffi import requests as creq
import json

s = creq.Session(impersonate="chrome")           # TLS Chrome untuk semua request
warm = s.get(PAGE, headers=BROWSER_HEADERS, timeout=20)   # warm-up -> dapat cookie
print("Warm-up halaman :", warm.status_code, "| cookie:", list(s.cookies.keys()))

resp = s.get(api_url(n=8), headers={"Referer": PAGE, "Accept": "application/json"}, timeout=20)
print("API setelah warm:", resp.status_code, "| content-type:", resp.headers.get("content-type"))

if resp.status_code == 200 and "json" in resp.headers.get("content-type", ""):
    produk = resp.json()["data"]["products"]
    print(f"\nTEMBUS — {len(produk)} produk asli:")
    for p in produk[:5]:
        print(f"  - {p['name'][:55]:55} | {p['price']['priceDisplay']}")

Warm-up halaman : 200 | cookie: ['__cf_bm', '_cfuvid']


API setelah warm: 200 | content-type: application/json

TEMBUS — 8 produk asli:
  - United Detroit 1121 Sepeda Gunung / MTB 27.5 12 Speed F | Rp4.200.000
  - Strider 14x Footrest - Black                            | Rp210.000
  - Strider 14x Clamp - Black                               | Rp50.000
  - United Detroit 1101 Sepeda Gunung / MTB 27.5 10 Speed F | Rp3.000.000
  - SEPEDA GUNUNG ELEMENT COYOTE MTB SPY 26 INCH 7 SPEED    | Rp1.695.000


**Kombinasi TLS-Chrome + cookie warm-up = tembus.** Inilah teknik yang umum dipakai di dunia
nyata untuk WAF berbasis cookie.

**Pencegahan (Blue Team):**
- **Managed Challenge / JS Challenge**: paksa eksekusi JavaScript sebelum cookie valid (cookie
  dari sekadar GET jadi tidak sah). `curl_cffi` tak menjalankan JS, jadi ini menghentikannya.
- **Ikat cookie ke fingerprint + IP + waktu**; TTL pendek; deteksi cookie dipakai-ulang lintas
  IP.
- **Rate-limit per cookie/IP** dan analitik perilaku (lihat Level 5).

### Level 4 — Browser sungguhan (Playwright/Selenium) *(konsep)*

Kalau WAF butuh **eksekusi JS** (managed challenge) atau perilaku manusiawi, penyerang naik ke
**browser headless** yang menjalankan JS sensor & me-render SPA. Sering ditambah **stealth**
(menyembunyikan `navigator.webdriver`, dll.).

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent=BROWSER_UA)
    page.goto(PAGE, wait_until="networkidle")     # jalankan JS sensor -> cookie valid
    # Tangkap respons API yang dipanggil halaman, atau baca DOM yang sudah ter-render:
    data = page.evaluate("() => window.fetch(API_URL).then(r => r.json())")
    browser.close()
```

**Pencegahan (Blue Team):**
- **Deteksi headless/otomasi**: `navigator.webdriver`, properti yang hilang, timing render.
- **Fingerprint kanvas/WebGL/AudioContext** + **biometrik perilaku** (gerak mouse, ritme).
- **CAPTCHA / managed challenge** saat skor risiko tinggi.

### Level 5 — Skala: rotasi proxy & UA *(konsep)*

Untuk volume besar, penyerang menyebar request lewat **banyak IP (proxy residensial)** dan
**rotasi UA/fingerprint** agar tidak kena rate-limit per-IP.

```python
# Scrapy: middleware proxy + rotasi UA (ilustrasi)
DOWNLOADER_MIDDLEWARES = {
    "scrapy_proxies.RandomProxy": 610,
    "myproject.middlewares.RandomUserAgent": 400,
}
ROTATING_PROXY_LIST = ["http://ip1:port", "http://ip2:port", ...]
```

**Pencegahan (Blue Team):**
- **Rate-limit & velocity checks** per IP/akun/cookie; **IP reputation** (blokir ASN datacenter /
  proxy dikenal).
- **Anomali volume/pola** (jam, urutan halaman tak manusiawi) → tantang/blokir.
- **Honeypot/honeytoken**: link/format harga tersembunyi yang hanya bot ambil → tandai.

## 3. Menggabungkannya di **Scrapy** (versi "hacky" yang benar-benar jalan)

Sekarang kita kemas teknik **Level 3** ke dalam Scrapy produksi memakai
[`scrapy-impersonate`](https://github.com/jxlil/scrapy-impersonate) (download handler berbasis
`curl_cffi` → TLS Chrome). Spider:

1. **Warm-up** ke halaman kategori (cookie `__cf_bm` otomatis disimpan cookie-jar Scrapy).
2. **Panggil API** `/backend/search` per halaman (pagination), parse JSON.
3. Normalisasi harga dengan **`price-parser`**, keluarkan sebagai **`Item`**.

> Catatan "hacky": kita set **`ROBOTSTXT_OBEY = False`** — secara teknis melawan `robots.txt`
> Blibli. **Ini sengaja**, sebagai bahan diskusi: inilah keputusan yang membuat aktivitas ini
> berpindah dari "scraping sopan" ke "wilayah abu-abu/ilegal". Di sistemmu sendiri, justru ini
> yang harus kamu antisipasi.

In [8]:
spider_code = r"""
import scrapy, json
from price_parser import Price

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
CAT = "AK-1000051"
PAGE = "https://www.blibli.com/c3/olahraga-sepeda/" + CAT


class ProdukItem(scrapy.Item):
    nama       = scrapy.Field()
    harga_teks = scrapy.Field()
    harga      = scrapy.Field()   # float dari price-parser
    brand      = scrapy.Field()
    rating     = scrapy.Field()
    terjual    = scrapy.Field()
    merchant   = scrapy.Field()
    url        = scrapy.Field()


class BlibliSpider(scrapy.Spider):
    name = "blibli"
    max_page = 2  # batasi demo: sopan & ringan

    custom_settings = {
        "ROBOTSTXT_OBEY": False,                 # <- keputusan 'hacky' (lihat catatan)
        "USER_AGENT": UA,
        "DEFAULT_REQUEST_HEADERS": {
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
            "sec-ch-ua": '\"Chromium\";v=\"124\", \"Google Chrome\";v=\"124\", \"Not-A.Brand\";v=\"99\"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '\"macOS\"',
        },
        # TLS Chrome via scrapy-impersonate (kunci lolos JA3)
        "DOWNLOAD_HANDLERS": {
            "https": "scrapy_impersonate.ImpersonateDownloadHandler",
            "http": "scrapy_impersonate.ImpersonateDownloadHandler",
        },
        "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
        "CONCURRENT_REQUESTS": 1,     # sopan
        "DOWNLOAD_DELAY": 1.0,        # jeda manusiawi
        "AUTOTHROTTLE_ENABLED": True,
        "LOG_LEVEL": "INFO",
        "FEED_EXPORT_ENCODING": "utf-8",
    }

    def api(self, page):
        return (f"https://www.blibli.com/backend/search/products"
                f"?categoryId={CAT}&page={page}&itemPerPage=12&channelId=web")

    async def start(self):
        # Level 3: warm-up halaman dulu untuk dapat cookie __cf_bm
        yield scrapy.Request(PAGE, meta={"impersonate": "chrome"}, callback=self.after_warmup)

    def after_warmup(self, response):
        self.logger.info(f"WARMUP status={response.status}")
        yield scrapy.Request(self.api(1), meta={"impersonate": "chrome", "page": 1},
                             headers={"Referer": PAGE, "Accept": "application/json"},
                             callback=self.parse_api)

    def parse_api(self, response):
        page = response.meta["page"]
        data = json.loads(response.text).get("data", {})
        produk = data.get("products", [])
        self.logger.info(f"APISTATUS page={page} status={response.status} produk={len(produk)}")

        for p in produk:
            harga = Price.fromstring(p["price"].get("priceDisplay", ""))
            rev = p.get("review") or {}
            it = ProdukItem()
            it["nama"]       = p.get("name")
            it["harga_teks"] = p["price"].get("priceDisplay")
            it["harga"]      = harga.amount_float
            it["brand"]      = p.get("brand")
            it["rating"]     = rev.get("absoluteRating")
            it["terjual"]    = p.get("soldCountTotal")
            it["merchant"]   = p.get("merchantName")
            it["url"]        = "https://www.blibli.com" + (p.get("url") or "")
            yield it

        # pagination terbatas
        if produk and page < self.max_page:
            nxt = page + 1
            yield scrapy.Request(self.api(nxt), meta={"impersonate": "chrome", "page": nxt},
                                 headers={"Referer": PAGE, "Accept": "application/json"},
                                 callback=self.parse_api)
"""

with open("/tmp/blibli_spider.py", "w") as f:
    f.write(spider_code)
print("Spider ditulis ke /tmp/blibli_spider.py")

Spider ditulis ke /tmp/blibli_spider.py


### Jalankan spider (via subprocess)

In [9]:
import subprocess, sys, os, time

OUT = "/tmp/blibli_hasil.json"
if os.path.exists(OUT):
    os.remove(OUT)

t0 = time.time()
res = subprocess.run(
    [sys.executable, "-m", "scrapy", "runspider", "/tmp/blibli_spider.py", "-O", OUT],
    capture_output=True, text=True, timeout=180,
)
durasi = time.time() - t0

for line in res.stderr.splitlines():
    if any(k in line for k in ["WARMUP", "APISTATUS", "item_scraped_count",
                                "robotstxt", "finish_reason"]):
        print(line.strip()[-140:])
print(f"\nReturncode: {res.returncode} | Durasi: {durasi:.1f}s")

2026-06-03 07:14:27 [blibli] INFO: WARMUP status=200
2026-06-03 07:14:31 [blibli] INFO: APISTATUS page=1 status=200 produk=12
2026-06-03 07:14:34 [blibli] INFO: APISTATUS page=2 status=200 produk=12
'finish_reason': 'finished',
'item_scraped_count': 24,

Returncode: 0 | Durasi: 7.9s


### Hasil → `pandas` (siap cleaning/DB)

In [10]:
import pandas as pd, json

with open("/tmp/blibli_hasil.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("Jumlah produk:", len(df))
print("\nTipe data:")
print(df.dtypes)
if len(df):
    print(f"\nHarga: termurah Rp{df['harga'].min():,.0f} | "
          f"termahal Rp{df['harga'].max():,.0f} | rata-rata Rp{df['harga'].mean():,.0f}")
df[["nama", "harga", "brand", "rating", "terjual", "merchant"]].head(10)

Jumlah produk: 24

Tipe data:
nama              str
harga_teks        str
harga         float64
brand             str
rating        float64
terjual        object
merchant          str
url               str
dtype: object

Harga: termurah Rp20,300 | termahal Rp4,320,000 | rata-rata Rp1,928,885


,nama,harga,brand,rating,terjual,merchant
0,United Detroit 1121 Sepeda Gunung / MTB 27.5 1...,4200000.0,United Bike,0.0,None,Serba Sepeda
1,Strider 14x Footrest - Black,210000.0,Strider,0.0,None,Strider Bikes
2,Strider 14x Clamp - Black,50000.0,Strider,0.0,None,Strider Bikes
3,United Detroit 1101 Sepeda Gunung / MTB 27.5 1...,3000000.0,United Bike,0.0,None,Serba Sepeda
4,SEPEDA GUNUNG ELEMENT COYOTE MTB SPY 26 INCH 7...,1695000.0,Element,0.0,None,tokohappybike Flagship Store
5,UNO Seatpost Sepeda SP358 31.6 400mm Bahan All...,379620.0,UNO,0.0,None,tokohappybike Flagship Store
6,Element Sepeda Hybrid Jasper Ukuran 700C 21 Sp...,2350000.0,ELEMENT BIKE,0.0,None,Toko Dunia Sepeda
7,Element Montreal URB Ukuran 700C Shimano Cues ...,4320000.0,Element,0.0,None,Serba Sepeda
8,SEPEDA HYBRID JASPER ELEMENT HYDRAULIC 8+ 700C...,2750000.0,ELEMENT BIKE,0.0,None,Toko Dunia Sepeda
9,Sepeda Listrik Uwinfly D66A Terbaru 2025 Anti ...,4150000.0,Uwinfly,4.7,None,U-WINFLY INDONESIA Flagship Store


## 4. Blue Team — Pertahanan Berlapis (ringkasan)

Pelajaran intinya: **tidak ada satu peluru perak.** Pertahanan yang baik = banyak lapisan,
masing-masing menaikkan biaya penyerang.

| Teknik penyerang (merah) | Cara mendeteksi/mencegah (biru) |
|---|---|
| UA non-browser / kosong (L0) | Blokir/curigai UA aneh; tapi jangan andalkan ini saja |
| Header browser palsu (L1) | *Header anomaly detection* (konsistensi `sec-ch-ua` vs UA, urutan header) |
| TLS impersonate / JA3 palsu (L2) | **Fingerprint JA3/JA4**; tolak bila TLS≠UA yang diklaim |
| Warm-up cookie WAF (L3) | **Managed/JS challenge** (wajib eksekusi JS); ikat cookie ke fingerprint+IP; TTL pendek |
| Browser headless + stealth (L4) | Deteksi `navigator.webdriver`, fingerprint kanvas/WebGL, **biometrik perilaku**, CAPTCHA |
| Rotasi proxy & UA, skala (L5) | **Rate-limit & velocity**, **IP reputation** (blokir ASN datacenter), anomali volume |
| Semua di atas | **Honeytoken**, monitoring & alerting, autentikasi+penandatanganan request untuk API sensitif |

**Yang sudah dilakukan Blibli dengan baik:** SPA + Cloudflare bot-management + `robots.txt`
melarang API + fingerprint TLS. **Yang bisa diperkuat:** managed challenge wajib-JS pada
endpoint data (mematahkan Level 3 di notebook ini), serta rate-limit per-cookie yang ketat.

## Penutup

- **"Bisa di-scrape" ≠ "boleh di-scrape".** Kita berhasil menembus secara teknis, tapi jalur
  datanya `DILARANG` `robots.txt` — di dunia nyata pakai **API resmi / izin / data berlisensi**.
- Nilai edukatifnya: kamu kini paham **rantai serangan (L0→L5)** dan **pertahanan tiap lapis**,
  sehingga bisa **mengamankan** API/situsmu sendiri.
- Teknik & pustaka di sini (`curl_cffi`, `scrapy-impersonate`, Playwright) bersifat **dual-use** —
  gunakan secara etis dan legal.